# Híbrido (ML-KEM-768 + X25519): Comparativo Pareado de Handshake

Este notebook orquestra os testes comparativos entre ML-KEM-768, X25519 e o modo Híbrido (`mlkem768x25519-sha256`), baixando o código do GitHub e salvando os resultados no Drive.

## 1. Configuração e Clonagem do Repositório

In [ ]:
#@title ⚙️ Configuração do Repositório e Google Drive { display-mode: "form" }
#@markdown Preencha com o link do seu repositório, branch e diretório de destino no Google Drive:

REPO_URL = "https://github.com/nicolasvessel/utfpr-pqc.git"  #@param {type:"string"}
BRANCH_NAME = "teste-x25519"  #@param {type:"string"}
SUBDIR = "3_Hibrido"  #@param {type:"string"}
DRIVE_PROJ = "/content/drive/MyDrive/pqc_benchmarks/Hibrido"  #@param {type:"string"}
WORK = "/content/work"
TARGET_DIR = f"{WORK}/{SUBDIR}" if SUBDIR else WORK

# === MONTAGEM DO DRIVE E CLONAGEM ===
from google.colab import drive
import os

print("1. Montando Google Drive...")
drive.mount('/content/drive')

print("2. Clonando repositório...")
%cd /content
!rm -rf {WORK}
!git clone -b {BRANCH_NAME} {REPO_URL} {WORK}

%cd {TARGET_DIR}
!chmod +x dependencias/run_all.sh
print("Ambiente configurado com sucesso!")

## 2. Orquestração Auditável (run_all.sh)
O script abaixo cuida de toda a configuração. Ele irá automaticamente:
- Verificar e forçar a instalação do Go 1.26.0
- Travar as dependências (`go mod verify`)
- Auditar e registrar o ambiente (CPU, AVX2, BMI2, ADX, RAM)
- Tirar hash SHA-256 das fontes e binários compilados
- Executar o modelo estatístico intercalado (ABBA)

In [ ]:
!bash dependencias/run_all.sh

## 3. Análise dos Resultados e Comprovação

In [ ]:
import pandas as pd
from IPython.display import display

try:
    df = pd.read_csv(f'{TARGET_DIR}/resultados/03_matematica_pprof.csv')
    display(df)
except FileNotFoundError:
    print("Arquivo CSV não encontrado. Houve uma falha na execução do benchmark.")

In [ ]:
!cat {TARGET_DIR}/resultados/02_benchstat_speedup.txt

In [ ]:
# Conferir a estabilidade da máquina virtual do Google durante os testes (excluindo warm-up round 0 e filtrando cpu0)
try:
    log = pd.read_csv(f'{TARGET_DIR}/resultados/04_telemetria_ruido.csv')
    log['round'] = pd.to_numeric(log['round'], errors='coerce')
    log['steal_delta'] = pd.to_numeric(log['steal_after'], errors='coerce') - pd.to_numeric(log['steal_before'], errors='coerce')
    log_valid = log[(log['round'] > 0) & (log['cpu_id'] == 'cpu0')]
    display(log_valid.groupby('binary')['steal_delta'].describe())
except Exception as e:
    print("Não foi possível analisar 04_telemetria_ruido.csv:", e)

## 4. Persistência de Dados (Google Drive)
Salva a pasta de resultados no seu Drive.

In [ ]:
import os

# Copiar os resultados para o Google Drive
print("Copiando resultados para o Google Drive...")
!mkdir -p "{DRIVE_PROJ}"
!rm -rf "{DRIVE_PROJ}/resultados"
!cp -r {TARGET_DIR}/resultados "{DRIVE_PROJ}/"
!ls -la "{DRIVE_PROJ}/resultados"
print("Salvo com sucesso no Drive!")